# Encode and decode ED sentences with the handwriting-aware tokenizers

For one sentence this notebook shows:

1. **Encoding** — how a handwriting-aware tokenizer splits the text into tokens, and which ID each token gets;
2. **Decoding** — how those IDs are turned back into exactly the same text;
3. a **comparison** of the three handwriting-aware tokenizers on the same sentence.

Only the frozen tokenizer files are used: no model, checkpoint or GPU. Pick a real ED label (needs the extracted ED data) or type your own text.

| Tokenizer | How it splits text |
| --- | --- |
| Handwriting bigram | Scans left to right; if the next two characters are a vocabulary bigram it takes them as one token, otherwise one character. |
| Handwriting BPE | Starts from single characters and applies the learned merges in their learned order. |
| Handwriting Unigram | Chooses the split with the highest total score of its learned pieces. |

Every tokenizer has 224 classes: ID 0 is the CTC blank, IDs 1–78 are the ED characters, and IDs 79–223 are the 145 learned pieces.

In [1]:
from IPython.display import Markdown, display

from loguru import logger
from tva.ed_analysis import load_records, load_tokenizer, show, show_table

logger.disable('tva')  # hide routine 'tokenizer loaded' messages

# ---- Settings: change these, then run all cells ----
SOURCE = 'ed'                 # 'ed' = a real ED label, 'text' = TEXT below
SPLIT, PART, FOLD, INDEX = 'wd', 'val', 0, 100
TEXT = 'The bookkeeper thinks about the weather.'
TOKENIZER = 'handwriting_bigram'  # 'handwriting_bigram', 'handwriting_bpe' or 'handwriting_unigram'

NAMES = ['handwriting_bigram', 'handwriting_bpe', 'handwriting_unigram']
tokenizers = {name: load_tokenizer(name) for name in NAMES}

if SOURCE == 'ed':
    record = load_records(SPLIT, FOLD, PART)[INDEX]
    text = record['label']
    print(f"ED {SPLIT.upper()} fold {FOLD} {PART} sample {INDEX} (record {record['id']}, writer {record['id_writer']})")
else:
    text = TEXT
print('Sentence:', repr(text))

ED WD fold 0 val sample 100 (record 773, writer 6230)
Sentence: 'Be sure to pack some sunscreen!'


## 1. Encoding

Each row is one token. *Learned piece* means a multi-character token from the vocabulary; *single character* is the fallback. The line below the table shows the same split with `|` between tokens.

In [2]:
tokenizer = tokenizers[TOKENIZER]
segments = tokenizer.segment(text)['segments']
ids = tokenizer.encode(text)
show_table([{'#': n, 'Position': s['start'] if s['end'] - s['start'] == 1 else f"{s['start']}–{s['end'] - 1}", 'Token': show(s['token']), 'ID': i,
             'Type': 'learned piece' if len(s['token']) > 1 else 'single character'}
            for n, (s, i) in enumerate(zip(segments, ids), 1)])
pieces = [s['token'] for s in segments]
print('Split:', '|'.join(pieces))
print('IDs:  ', ids)
print(f'{len(text)} characters -> {len(ids)} tokens ({sum(len(p) > 1 for p in pieces)} learned pieces)')

| # | Position | Token | ID | Type |
| --- | --- | --- | --- | --- |
| 1 | 0–1 | 'Be' | 86 | learned piece |
| 2 | 2 | ' ' | 1 | single character |
| 3 | 3 | 's' | 71 | single character |
| 4 | 4–5 | 'ur' | 218 | learned piece |
| 5 | 6 | 'e' | 57 | single character |
| 6 | 7 | ' ' | 1 | single character |
| 7 | 8–9 | 'to' | 209 | learned piece |
| 8 | 10 | ' ' | 1 | single character |
| 9 | 11 | 'p' | 68 | single character |
| 10 | 12 | 'a' | 53 | single character |
| 11 | 13–14 | 'ck' | 127 | learned piece |
| 12 | 15 | ' ' | 1 | single character |
| 13 | 16 | 's' | 71 | single character |
| 14 | 17–18 | 'om' | 190 | learned piece |
| 15 | 19 | 'e' | 57 | single character |
| 16 | 20 | ' ' | 1 | single character |
| 17 | 21 | 's' | 71 | single character |
| 18 | 22–23 | 'un' | 217 | learned piece |
| 19 | 24 | 's' | 71 | single character |
| 20 | 25–26 | 'cr' | 130 | learned piece |
| 21 | 27–28 | 'ee' | 140 | learned piece |
| 22 | 29 | 'n' | 66 | single character |
| 23 | 30 | '!' | 2 | single character |

Split: Be| |s|ur|e| |to| |p|a|ck| |s|om|e| |s|un|s|cr|ee|n|!
IDs:   [86, 1, 71, 218, 57, 1, 209, 1, 68, 53, 127, 1, 71, 190, 57, 1, 71, 217, 71, 130, 140, 66, 2]
31 characters -> 23 tokens (8 learned pieces)


## 2. Decoding

Decoding looks up the token of every ID and joins the tokens. Blank (ID 0) would become an empty string; the encoder never produces it. A correct tokenizer gives back exactly the original sentence.

In [3]:
looked_up = [tokenizer.idx_token[i] for i in ids]
decoded = tokenizer.decode(ids)
show_table([{'ID': i, 'Token': show(t)} for i, t in zip(ids, looked_up)])
print('Joined tokens:', repr(''.join(looked_up)))
print('Decoded text: ', repr(decoded))
print('Original text:', repr(text))
print('Round trip:', 'identical ✅' if decoded == text else 'DIFFERENT ❌')

| ID | Token |
| --- | --- |
| 86 | 'Be' |
| 1 | ' ' |
| 71 | 's' |
| 218 | 'ur' |
| 57 | 'e' |
| 1 | ' ' |
| 209 | 'to' |
| 1 | ' ' |
| 68 | 'p' |
| 53 | 'a' |
| 127 | 'ck' |
| 1 | ' ' |
| 71 | 's' |
| 190 | 'om' |
| 57 | 'e' |
| 1 | ' ' |
| 71 | 's' |
| 217 | 'un' |
| 71 | 's' |
| 130 | 'cr' |
| 140 | 'ee' |
| 66 | 'n' |
| 2 | '!' |

Joined tokens: 'Be sure to pack some sunscreen!'
Decoded text:  'Be sure to pack some sunscreen!'
Original text: 'Be sure to pack some sunscreen!'
Round trip: identical ✅


## 3. Compare the three handwriting-aware tokenizers

Same sentence, three tokenizers. Fewer tokens means longer pieces were used (stronger compression).

In [4]:
rows = []
for name, t in tokenizers.items():
    token_ids = t.encode(text)
    split = [t.idx_token[i] for i in token_ids]
    rows.append({'Tokenizer': name, 'Tokens': len(token_ids),
                 'Learned pieces': sum(len(p) > 1 for p in split),
                 'Characters per token': f'{len(text) / len(token_ids):.2f}',
                 'Split': '|'.join(split),
                 'Round trip': '✅' if t.decode(token_ids) == text else '❌'})
show_table(rows)

| Tokenizer | Tokens | Learned pieces | Characters per token | Split | Round trip |
| --- | --- | --- | --- | --- | --- |
| handwriting_bigram | 23 | 8 | 1.35 | Be\| \|s\|ur\|e\| \|to\| \|p\|a\|ck\| \|s\|om\|e\| \|s\|un\|s\|cr\|ee\|n\|! | ✅ |
| handwriting_bpe | 24 | 7 | 1.29 | B\|e\| \|s\|u\|re\| \|to\| \|p\|a\|ck\| \|s\|o\|me\| \|s\|un\|s\|c\|re\|en\|! | ✅ |
| handwriting_unigram | 24 | 7 | 1.29 | B\|e\| \|s\|u\|re\| \|to\| \|p\|a\|ck\| \|s\|o\|me\| \|s\|un\|s\|c\|re\|en\|! | ✅ |

## 4. Optional: check a whole fold

Encodes and decodes every label in the chosen split, part and fold, and counts round-trip failures. Needs the extracted ED data.

In [5]:
records = load_records(SPLIT, FOLD, PART)
rows = []
for name, t in tokenizers.items():
    encoded = [t.encode(r['label']) for r in records]
    failures = sum(t.decode(e) != r['label'] for e, r in zip(encoded, records))
    total = sum(map(len, encoded))
    rows.append({'Tokenizer': name, 'Sentences': len(records), 'Round-trip failures': failures,
                 'Tokens per sentence': f'{total / len(records):.1f}',
                 'Characters per token': f"{sum(len(r['label']) for r in records) / total:.2f}"})
show_table(rows)

| Tokenizer | Sentences | Round-trip failures | Tokens per sentence | Characters per token |
| --- | --- | --- | --- | --- |
| handwriting_bigram | 503 | 0 | 33.6 | 1.26 |
| handwriting_bpe | 503 | 0 | 33.0 | 1.28 |
| handwriting_unigram | 503 | 0 | 33.4 | 1.26 |